# 🎯 5. Planning and Reasoning Patterns

Reasoning is what makes an agent *intelligent*. The LLM's ability to think step-by-step, decompose problems, and evaluate its own plans is the core differentiator from simple automation.

In this notebook:

1. **Chain-of-Thought (CoT)** — step-by-step reasoning
2. **Task decomposition** — breaking complex problems into sub-tasks
3. **Planner-Executor pattern** — separate planning from execution
4. **Tree of Thought (ToT)** — exploring multiple reasoning paths
5. **Graph of Thought (GoT)** — combining multiple perspectives
6. **Self-reflection** — agents that evaluate their own output
7. **Reasoning topology comparison** — when to use each pattern

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

LLM_MODEL   = "gpt-4o-mini"

client = OpenAI()
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
print(f'Model: {LLM_MODEL}')

Model: gpt-4o-mini


## 5.1 Chain-of-Thought (CoT) Reasoning

**Chain-of-Thought** prompting makes the LLM show its reasoning step by step. This significantly improves accuracy on complex tasks.

```
Without CoT: "The answer is 42"
With CoT:    "First, I need to... Then... Therefore... The answer is 42"
```

For agents, CoT is built into the **Thought** step of the agent loop.

In [3]:
# Compare: without vs with Chain-of-Thought
from IPython.display import display, Markdown

problem = (
    'A store has 3 shelves. The first shelf has twice as many books as the second. '
    'The third shelf has 5 fewer books than the first. '
    'If the total is 55 books, how many books are on each shelf?'
)

# Without CoT
r1 = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': f'Answer directly: {problem} (NO LaTeX)'}],
    temperature=0
)
display(Markdown('\n ### ------------WITHOUT CoT------------'))
display(Markdown(r1.choices[0].message.content))

# With CoT
r2 = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': f'Think step by step and show your work: {problem} (NO LaTeX)'}],
    temperature=0
)
display(Markdown('\n ### ------------WITH CoT------------'))
display(Markdown(r2.choices[0].message.content))


 ### ------------WITHOUT CoT------------

Let the number of books on the second shelf be x. Then, the first shelf has 2x books, and the third shelf has (2x - 5) books. 

The equation for the total number of books is:
x + 2x + (2x - 5) = 55.

Combining the terms gives:
5x - 5 = 55.

Adding 5 to both sides:
5x = 60.

Dividing by 5:
x = 12.

Now, we can find the number of books on each shelf:
- Second shelf: 12 books.
- First shelf: 2 * 12 = 24 books.
- Third shelf: 24 - 5 = 19 books.

So, the shelves have:
First shelf: 24 books,
Second shelf: 12 books,
Third shelf: 19 books.


 ### ------------WITH CoT------------

Let's define the number of books on each shelf using variables:

- Let the number of books on the second shelf be x.
- Then, the first shelf has twice as many books as the second shelf, so the number of books on the first shelf is 2x.
- The third shelf has 5 fewer books than the first shelf, so the number of books on the third shelf is 2x - 5.

Now, we can express the total number of books on all three shelves:

Total books = (books on first shelf) + (books on second shelf) + (books on third shelf)

Substituting the expressions we defined:

Total books = (2x) + (x) + (2x - 5)

Now, we can combine like terms:

Total books = 2x + x + 2x - 5
Total books = 5x - 5

We know that the total number of books is 55, so we can set up the equation:

5x - 5 = 55

Next, we will solve for x:

1. Add 5 to both sides of the equation:
   5x - 5 + 5 = 55 + 5
   5x = 60

2. Now, divide both sides by 5:
   5x / 5 = 60 / 5
   x = 12

Now that we have the value of x, we can find the number of books on each shelf:

- Books on the second shelf (x) = 12
- Books on the first shelf (2x) = 2 * 12 = 24
- Books on the third shelf (2x - 5) = 24 - 5 = 19

Finally, let's summarize the number of books on each shelf:

- First shelf: 24 books
- Second shelf: 12 books
- Third shelf: 19 books

To verify, we can check the total:

24 + 12 + 19 = 55

The calculations are correct. Therefore, the number of books on each shelf is:

- First shelf: 24 books
- Second shelf: 12 books
- Third shelf: 19 books

## 5.2 Task Decomposition — The Planner-Executor Pattern

Complex tasks should be broken into smaller, manageable sub-tasks. The **Planner-Executor** pattern separates:

- **Planner**: LLM creates a structured plan
- **Executor**: Code executes each step

This is safer than letting the agent improvise — the plan can be reviewed before execution.

In [4]:
from pydantic import BaseModel, Field
from typing import List, Literal

class SubTask(BaseModel):
    """A single step in the execution plan"""
    step_number: int = Field(description="Order of execution")
    description: str = Field(description="What this step does")
    tool_needed: str = Field(description="Tool to use: search, calculate, write or none")
    depends_on: List[int] = Field(default=[], description="Step numbers this depends on")


class ExecutionPlan(BaseModel):
    """A structured plan for completing a complex task"""
    goal: str = Field(description="The main goal")
    subtasks: List[SubTask] = Field(description="Ordered list of subtasks")
    estimated_steps: int = Field(description="Total number of steps")

planner = llm.with_structured_output(ExecutionPlan)

plan = planner.invoke(
    'Create a plan to: Research the top 3 AI Agent Frameworks'
    "compare their features, and write a recommendation report"
)

print(f"Goal: {plan.goal}")
print(f"Steps: {plan.estimated_steps}\n")

for task in plan.subtasks:
    deps = f" depends on: {task.depends_on}" if task.depends_on else ''
    print(f"  {task.step_number}. [{task.tool_needed}] {task.description}{deps}")



Goal: Research the top 3 AI Agent Frameworks, compare their features, and write a recommendation report.
Steps: 5

  1. [search] Identify the top 3 AI Agent Frameworks based on popularity and usage.
  2. [search] Gather detailed information about the features of each of the identified frameworks. depends on: [1]
  3. [none] Analyze the gathered information to compare the features of the frameworks side by side. depends on: [2]
  4. [write] Draft a recommendation report based on the analysis, highlighting the strengths and weaknesses of each framework. depends on: [3]
  5. [write] Review and edit the recommendation report for clarity and completeness. depends on: [4]


In [5]:
def execute_plan(plan: ExecutionPlan) -> dict:
    """Execute a plan step by step, collection results"""
    results = {}

    for task in plan.subtasks:
        display(Markdown(f"\n\n#### **Executing Step {task.step_number}**: {task.description}"))

        # Check dependencies
        dep_context = ''
        for dep in task.depends_on:
            if dep in results:
                dep_context += f"\nResult from step {dep}: {results[dep]}"
        
        # Execute based on loop type
        if task.tool_needed == "none":
            result = f"**Step {task.step_number}** completed (no tool needed)"
        else:
            # In production this would call real tools !!!
            prompt = f"#### Execute this task: {task.description}"
            if dep_context:
                prompt += f"\n\nContext from previous steps: {dep_context}"

            response = llm.invoke([
                SystemMessage(content="Complete the task concisely. Use the content provided"),
                HumanMessage(content=prompt)
            ]) 
            result = response.content
        
        results[task.step_number] = result
        display(Markdown(f"#### Result: {result}"))
    
    return results

results = execute_plan(plan)
display(Markdown(f"\n\n"))
display(Markdown(f"### Plan completed! {len(results)} steps executed"))



#### **Executing Step 1**: Identify the top 3 AI Agent Frameworks based on popularity and usage.

#### Result: The top 3 AI Agent Frameworks based on popularity and usage are:

1. **Rasa** - An open-source framework for building conversational AI and chatbots, known for its flexibility and strong community support.
2. **Microsoft Bot Framework** - A comprehensive framework for building and connecting intelligent bots, widely used in enterprise applications.
3. **OpenAI Gym** - A toolkit for developing and comparing reinforcement learning algorithms, popular in research and academic settings.



#### **Executing Step 2**: Gather detailed information about the features of each of the identified frameworks.

#### Result: ### Features of the Identified AI Agent Frameworks

1. **Rasa**
   - **Open Source**: Fully open-source, allowing for customization and community contributions.
   - **Natural Language Understanding (NLU)**: Provides robust NLU capabilities to understand user intents and extract entities.
   - **Dialogue Management**: Uses machine learning to manage conversations and maintain context.
   - **Custom Actions**: Supports the creation of custom actions to integrate with external APIs and databases.
   - **Multi-language Support**: Capable of handling multiple languages, making it versatile for global applications.
   - **Interactive Learning**: Allows for interactive training, enabling developers to improve models based on real user interactions.
   - **Integration**: Easily integrates with messaging platforms like Slack, Facebook Messenger, and more.
   - **Strong Community**: Active community support with extensive documentation and tutorials.

2. **Microsoft Bot Framework**
   - **Comprehensive SDK**: Offers a rich SDK for building bots across various platforms, including web, mobile, and desktop.
   - **Azure Integration**: Seamlessly integrates with Azure services for enhanced capabilities like natural language processing and machine learning.
   - **Multi-channel Support**: Deploys bots across multiple channels such as Microsoft Teams, Skype, and other messaging platforms.
   - **Bot Framework Composer**: A visual authoring tool for building and managing bot dialogues without extensive coding.
   - **Adaptive Cards**: Supports rich interactive content through adaptive cards, enhancing user engagement.
   - **Enterprise Features**: Includes features like authentication, state management, and telemetry for enterprise-level applications.
   - **Extensive Documentation**: Comprehensive resources and tutorials available for developers.

3. **OpenAI Gym**
   - **Reinforcement Learning Toolkit**: Provides a variety of environments for developing and testing reinforcement learning algorithms.
   - **Standardized API**: Offers a consistent interface for different environments, making it easier to switch between them.
   - **Diverse Environments**: Includes classic control tasks, Atari games, and robotics simulations for varied testing scenarios.
   - **Integration with Libraries**: Compatible with popular machine learning libraries like TensorFlow and PyTorch for model training.
   - **Benchmarking**: Facilitates benchmarking of algorithms against standard environments to evaluate performance.
   - **Community Contributions**: Open to community contributions, allowing users to create and share new environments.
   - **Educational Resources**: Provides tutorials and documentation to help users understand reinforcement learning concepts.

These frameworks cater to different aspects of AI development, from conversational agents to reinforcement learning, each with unique features that support various use cases.



#### **Executing Step 3**: Analyze the gathered information to compare the features of the frameworks side by side.

...



#### **Executing Step 4**: Draft a recommendation report based on the analysis, highlighting the strengths and weaknesses of each framework.

#### Result: ### Recommendation Report: Analysis of Frameworks

#### Introduction
This report evaluates various frameworks based on their strengths and weaknesses, providing recommendations for their application in relevant contexts.

#### Framework Analysis

1. **Framework A**
   - **Strengths:**
     - High adaptability to different project sizes and scopes.
     - Strong community support and extensive documentation.
     - Proven track record in enhancing team collaboration.
   - **Weaknesses:**
     - Steeper learning curve for new users.
     - May require additional resources for implementation.

2. **Framework B**
   - **Strengths:**
     - User-friendly interface, making it accessible for beginners.
     - Quick deployment and integration with existing systems.
     - Strong focus on security features.
   - **Weaknesses:**
     - Limited customization options.
     - Less effective for large-scale projects.

3. **Framework C**
   - **Strengths:**
     - Excellent performance metrics and scalability.
     - Comprehensive analytics tools for data-driven decision-making.
     - Strong emphasis on best practices and standards.
   - **Weaknesses:**
     - Higher cost of implementation and maintenance.
     - Requires specialized knowledge for optimal use.

#### Recommendations
- **For Small to Medium Projects:** Framework B is recommended due to its user-friendly nature and quick deployment.
- **For Large Scale Projects:** Framework C is advisable for its scalability and performance metrics, despite the higher costs.
- **For Teams Seeking Flexibility:** Framework A is ideal for teams that require adaptability and strong community support.

#### Conclusion
Each framework has unique strengths and weaknesses that cater to different project needs. The recommendations provided aim to guide decision-makers in selecting the most suitable framework based on their specific requirements.



#### **Executing Step 5**: Review and edit the recommendation report for clarity and completeness.

#### Result: ### Recommendation Report: Analysis of Frameworks

#### Introduction
This report evaluates various frameworks, highlighting their strengths and weaknesses, and provides recommendations for their application in relevant contexts.

#### Framework Analysis

1. **Framework A**
   - **Strengths:**
     - Highly adaptable to various project sizes and scopes.
     - Strong community support and extensive documentation.
     - Proven track record in enhancing team collaboration.
   - **Weaknesses:**
     - Steeper learning curve for new users.
     - May require additional resources for implementation.

2. **Framework B**
   - **Strengths:**
     - User-friendly interface, making it accessible for beginners.
     - Quick deployment and seamless integration with existing systems.
     - Strong focus on security features.
   - **Weaknesses:**
     - Limited customization options.
     - Less effective for large-scale projects.

3. **Framework C**
   - **Strengths:**
     - Excellent performance metrics and scalability.
     - Comprehensive analytics tools for data-driven decision-making.
     - Strong emphasis on best practices and standards.
   - **Weaknesses:**
     - Higher cost of implementation and maintenance.
     - Requires specialized knowledge for optimal use.

#### Recommendations
- **For Small to Medium Projects:** Framework B is recommended due to its user-friendly nature and quick deployment capabilities.
- **For Large Scale Projects:** Framework C is advisable for its scalability and performance metrics, despite the higher costs associated with implementation and maintenance.
- **For Teams Seeking Flexibility:** Framework A is ideal for teams that require adaptability and benefit from strong community support.

#### Conclusion
Each framework presents unique strengths and weaknesses that cater to different project needs. The recommendations provided aim to assist decision-makers in selecting the most suitable framework based on their specific requirements.

### Plan completed! 5 steps executed

## 5.3 Implementing the Planner-Executor Pattern

## 5.4 Tree of Thought (ToT) — Exploring Multiple Paths

Instead of a single chain of reasoning, **Tree of Thought** explores multiple reasoning branches and selects the best:

<img src="images/tree-of-thought.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

**Key operations**: Branch → Score → Prune → Select best

In [1]:
from IPython.display import display, Markdown


def stream_markdown(messages, title: str = None) -> str:
    """
    Streams an LLM response into a live-updating Markdown display.
    Returns the final accumulated text.
    """

    if title:
        display(Markdown(f"## {title}"))

    streamed_display = display(Markdown(""), display_id=True)

    full_text = ""

    for chunk in llm.stream(messages):
        # For ChatModels, chunk usually has .content
        token = chunk.content if hasattr(chunk, "content") else str(chunk)

        full_text += token
        streamed_display.update(Markdown(full_text))

    return full_text

In [6]:
from IPython.display import display, Markdown

def simple_tree_of_thought_2_streaming(problem: str, num_branches: int = 3) -> str:
    """
    Simplified Tree of Thought with streaming output:
    1. Generate multiple candidate approaches.
    2. Evaluate each approach.
    3. Select the best approach.
    4. Develop the final answer.

    This is a one-level Tree of Thought, not a full recursive tree search.
    """

    # Step 1: BRANCH
    branch_prompt = f"""
Generate {num_branches} different solution approaches for the following problem.

For each approach, include:
1. Approach name
2. Key strategy
3. Initial reasoning

Problem:
{problem}
"""

    branches = stream_markdown(
        [
            SystemMessage(content="You generate diverse solution strategies."),
            HumanMessage(content=branch_prompt)
        ],
        title="Branches Generated"
    )

    # Step 2: EVALUATE
    score_prompt = f"""
Evaluate each approach using the following criteria:

- Feasibility: Can this approach work?
- Efficiency: Is it practical at scale?
- Completeness: Does it solve the full problem?
- Risks: What could go wrong?

Give each approach a score from 1 to 10.

Then choose the best approach and explain why.

Approaches:
{branches}
"""

    evaluation = stream_markdown(
        [
            SystemMessage(content="You are a rigorous evaluator of solution strategies."),
            HumanMessage(content=score_prompt)
        ],
        title="Evaluation"
    )

    # Step 3: EXECUTE BEST PATH
    final_prompt = f"""
Based on the following evaluation, develop the winning approach into a complete solution.

Evaluation:
{evaluation}

Original problem:
{problem}
"""

    final_solution = stream_markdown(
        [
            SystemMessage(content="You develop the selected approach into a complete, practical solution."),
            HumanMessage(content=final_prompt)
        ],
        title="Final Solution"
    )

    return final_solution

In [7]:
result2 = simple_tree_of_thought_2_streaming(
    "Design a system to automatically categorize the prioritize customer support emails"
    "for a company that receives 10.000 emails per day"
)

## Branches Generated

### Approach 1: Machine Learning Classification

1. **Approach Name**: Supervised Learning Classifier
2. **Key Strategy**: Utilize a supervised machine learning model to classify and prioritize emails based on historical data.
3. **Initial Reasoning**: By training a model on a labeled dataset of past customer support emails, the system can learn to identify patterns and features that correlate with different categories and priority levels. This approach allows for continuous improvement as more data is collected, and it can adapt to changing customer needs over time.

---

### Approach 2: Rule-Based System

1. **Approach Name**: Heuristic Rule Engine
2. **Key Strategy**: Develop a set of predefined rules based on keywords, phrases, and sender information to categorize and prioritize emails.
3. **Initial Reasoning**: A rule-based system can quickly categorize emails using simple logic and keyword matching, which can be effective for straightforward cases. This approach is easy to implement and can be adjusted as needed without requiring extensive training data, making it suitable for immediate deployment.

---

### Approach 3: Hybrid System

1. **Approach Name**: Combined Machine Learning and Rule-Based System
2. **Key Strategy**: Integrate both machine learning and rule-based methods to leverage the strengths of each approach for better accuracy and efficiency.
3. **Initial Reasoning**: By combining the precision of machine learning with the speed and simplicity of rule-based categorization, the system can handle a wide variety of email types. The rule-based component can filter out obvious cases quickly, while the machine learning model can focus on more complex emails that require nuanced understanding, thus optimizing the overall processing time and accuracy.

## Evaluation

### Evaluation of Approaches

#### Approach 1: Machine Learning Classification

- **Feasibility**: 8/10
  - This approach can work effectively if there is sufficient labeled data for training. However, it requires a robust dataset and may struggle with edge cases or new types of emails that were not present in the training data.

- **Efficiency**: 7/10
  - While machine learning models can be efficient once trained, the initial training phase can be resource-intensive. Additionally, the model may require regular updates and retraining as new data comes in.

- **Completeness**: 8/10
  - This approach can address a wide range of email types and complexities, but it may not cover all scenarios, especially if the model is not well-trained on diverse data.

- **Risks**: 6/10
  - Risks include overfitting, reliance on the quality of training data, and potential biases in the model. Misclassifications can lead to poor customer experiences.

**Total Score**: 29/40

---

#### Approach 2: Rule-Based System

- **Feasibility**: 9/10
  - This approach is highly feasible as it can be implemented quickly with minimal resources. It does not require extensive data for training.

- **Efficiency**: 8/10
  - Rule-based systems can process emails quickly, especially for straightforward cases. However, they may become cumbersome as the number of rules increases.

- **Completeness**: 6/10
  - While effective for clear-cut cases, this approach may struggle with nuanced emails that require deeper understanding, leading to incomplete categorization.

- **Risks**: 7/10
  - Risks include the potential for false positives/negatives due to rigid rules and the need for ongoing maintenance to update rules as language and email types evolve.

**Total Score**: 30/40

---

#### Approach 3: Hybrid System

- **Feasibility**: 9/10
  - This approach is feasible as it combines two established methods, allowing for a more robust solution. However, it requires careful integration of both systems.

- **Efficiency**: 9/10
  - The hybrid system can efficiently handle a wide variety of emails by quickly filtering obvious cases with rules and applying machine learning for more complex ones, optimizing processing time.

- **Completeness**: 9/10
  - This approach is likely to be the most complete, as it can address both straightforward and complex emails effectively, leveraging the strengths of both methods.

- **Risks**: 5/10
  - Risks include the complexity of integration, potential conflicts between the two systems, and the need for ongoing maintenance to ensure both components work well together.

**Total Score**: 32/40

---

### Best Approach

**Chosen Approach**: Hybrid System

**Reasoning**: The hybrid system scores the highest overall due to its ability to combine the strengths of both machine learning and rule-based systems. It offers a balanced solution that can efficiently handle a wide range of email types while maintaining a high level of accuracy. The integration of both methods allows for quick processing of straightforward cases and a more nuanced understanding of complex emails, making it the most complete solution. Although it carries some risks related to integration, the benefits of improved efficiency and completeness outweigh these concerns, making it the best choice for categorizing and prioritizing emails.

## Final Solution

### Complete Solution: Hybrid Email Categorization and Prioritization System

#### Overview
The proposed solution is a Hybrid Email Categorization and Prioritization System that combines machine learning classification and a rule-based system to efficiently handle and categorize customer support emails. This system is designed to process approximately 10,000 emails per day, ensuring that emails are categorized accurately and prioritized based on urgency and complexity.

#### System Architecture
1. **Email Ingestion Module**
   - **Function**: Collects incoming emails from various sources (e.g., support tickets, direct emails).
   - **Technology**: Use an email server API (e.g., IMAP, SMTP) to fetch emails.

2. **Preprocessing Module**
   - **Function**: Cleans and preprocesses the email content for further analysis.
   - **Tasks**:
     - Remove signatures, disclaimers, and irrelevant content.
     - Normalize text (lowercasing, removing special characters).
     - Tokenization and stemming/lemmatization.
   - **Technology**: Python libraries such as NLTK or SpaCy.

3. **Rule-Based Categorization Module**
   - **Function**: Quickly categorizes straightforward emails based on predefined rules.
   - **Implementation**:
     - Define a set of rules based on keywords, phrases, and patterns (e.g., "refund", "technical issue").
     - Use regular expressions to match patterns.
   - **Output**: Assigns a preliminary category (e.g., Billing, Technical Support, General Inquiry).

4. **Machine Learning Classification Module**
   - **Function**: Classifies more complex emails that do not match the rules.
   - **Implementation**:
     - **Model Selection**: Use a supervised learning model (e.g., Random Forest, SVM, or a neural network).
     - **Training**: Train the model on a labeled dataset of historical emails.
     - **Features**: Use features such as TF-IDF vectors, email metadata (sender, subject), and sentiment analysis.
   - **Output**: Assigns a category and a confidence score.

5. **Prioritization Module**
   - **Function**: Prioritizes emails based on urgency and complexity.
   - **Implementation**:
     - Define criteria for prioritization (e.g., keywords indicating urgency, customer status).
     - Combine outputs from both the rule-based and ML modules to determine the final priority level (High, Medium, Low).
   - **Output**: Assigns a priority level to each email.

6. **Integration and Decision Module**
   - **Function**: Integrates outputs from both categorization modules and makes final decisions.
   - **Implementation**:
     - If the rule-based system provides a category, use it unless the ML model provides a higher confidence score for a different category.
     - Use a confidence threshold to determine if the ML classification should override the rule-based output.
   - **Output**: Final category and priority level for each email.

7. **Feedback Loop and Continuous Learning**
   - **Function**: Improves the system over time based on user feedback and new data.
   - **Implementation**:
     - Allow support agents to provide feedback on categorization accuracy.
     - Use this feedback to retrain the ML model periodically.
   - **Technology**: Store feedback in a database for analysis.

8. **User Interface**
   - **Function**: Provides a dashboard for support agents to view categorized and prioritized emails.
   - **Implementation**:
     - Develop a web-based interface using frameworks like React or Angular.
     - Display emails with their categories and priority levels, allowing agents to filter and sort as needed.

#### Technology Stack
- **Backend**: Python (Flask or Django for API), Scikit-learn for ML, NLTK/SpaCy for NLP.
- **Frontend**: React or Angular for the user interface.
- **Database**: PostgreSQL or MongoDB for storing emails, rules, and feedback.
- **Deployment**: Docker for containerization, AWS or Azure for cloud hosting.

#### Implementation Plan
1. **Phase 1: Requirements Gathering**
   - Collaborate with stakeholders to define rules and gather historical email data.

2. **Phase 2: Development**
   - Build the email ingestion, preprocessing, and categorization modules.
   - Develop the machine learning model and train it on historical data.

3. **Phase 3: Integration**
   - Integrate the rule-based and ML modules, and implement the prioritization logic.

4. **Phase 4: Testing**
   - Conduct thorough testing with a subset of emails to validate accuracy and performance.

5. **Phase 5: Deployment**
   - Deploy the system in a production environment and monitor performance.

6. **Phase 6: Feedback and Iteration**
   - Collect feedback from support agents and iterate on the system to improve accuracy and efficiency.

#### Conclusion
The Hybrid Email Categorization and Prioritization System is designed to efficiently manage a high volume of customer support emails, ensuring that they are categorized and prioritized effectively. By leveraging both rule-based and machine learning approaches, the system can handle a wide range of email types while continuously improving through user feedback. This solution not only enhances operational efficiency but also improves customer satisfaction by ensuring timely responses to urgent inquiries.

## 5.5 Self-Reflection — Agents That Check Their Own Work

Self-reflection is a powerful pattern where the agent **evaluates its own output** and iterates if needed:

<img src="images/self-refrection.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

In [8]:
from IPython.display import display, Markdown

def generate_with_reflection(task: str, max_iterations: int = 3) -> str:
    """
    Generate a response, evaluate it, and improve iteratively.
    Display each iteration using Markdown.
    """
    current_output = ""
    feedback = ""

    display(Markdown(f"# Reflection-based generation"))
    display(Markdown(f"**Task:** {task}"))

    for i in range(max_iterations):
        display(Markdown(f"---\n## Iteration {i + 1}"))

        # Generate or improve
        if i == 0:
            gen_prompt = f"Complete this task: {task}"
        else:
            gen_prompt = f"""
Improve this output based on the feedback:

Original output:
{current_output}

Feedback:
{feedback}
"""

        response = llm.invoke([
            HumanMessage(content=gen_prompt)
        ])

        current_output = response.content

        display(Markdown("### Generated Output"))
        display(Markdown(current_output))

        # Evaluate
        eval_response = llm.invoke([
            SystemMessage(content=(
                "Evaluate this output on a scale of 1-10. "
                "If score >= 8, say APPROVED. "
                "If score < 8, provide specific improvement feedback."
            )),
            HumanMessage(content=f"Task: {task}\n\nOutput:\n{current_output}")
        ])

        feedback = eval_response.content

        display(Markdown("### Evaluation"))
        display(Markdown(feedback))

        if "APPROVED" in feedback.upper():
            display(Markdown(f"## Approved after {i + 1} iteration(s)!"))
            return current_output

    display(Markdown("## Max iterations reached"))
    return current_output

In [10]:
result = generate_with_reflection(
    "Write a concise 3-point summary of why AI agents need memory systems."
)

# Reflection-based generation

**Task:** Write a concise 3-point summary of why AI agents need memory systems.

---
## Iteration 1

### Generated Output

1. **Contextual Understanding**: Memory systems enable AI agents to retain and recall past interactions, allowing them to understand context and provide more relevant and personalized responses over time.

2. **Learning and Adaptation**: By storing experiences and information, memory systems facilitate continuous learning, enabling AI agents to adapt their behavior and improve performance based on previous encounters and feedback.

3. **Efficiency and Resource Management**: Memory systems help AI agents manage information more effectively, reducing the need to repeatedly process the same data and enhancing overall efficiency in decision-making and task execution.

### Evaluation

Score: 9

APPROVED.

## Approved after 1 iteration(s)!

In [11]:
result = generate_with_reflection(
"""
Write exactly 3 bullet points explaining why autonomous software systems need long-term context.

Strict output rules:
1. Output exactly 3 bullet points.
2. Each bullet must contain exactly 7 words.
3. Each bullet must start with a different verb.
4. Do not use these words: AI, agent, agents, memory, memories, remember, context.
5. Do not add a title, introduction, explanation, or conclusion.

Evaluator instructions:
- Check every rule explicitly.
- If even one rule is violated, the score must be 7 or lower.
- Only say APPROVED if all rules are satisfied.
- Give specific feedback about which rule failed.
"""
)

# Reflection-based generation

**Task:** 
Write exactly 3 bullet points explaining why autonomous software systems need long-term context.

Strict output rules:
1. Output exactly 3 bullet points.
2. Each bullet must contain exactly 7 words.
3. Each bullet must start with a different verb.
4. Do not use these words: AI, agent, agents, memory, memories, remember, context.
5. Do not add a title, introduction, explanation, or conclusion.

Evaluator instructions:
- Check every rule explicitly.
- If even one rule is violated, the score must be 7 or lower.
- Only say APPROVED if all rules are satisfied.
- Give specific feedback about which rule failed.


---
## Iteration 1

### Generated Output

- Enhance decision-making through historical data analysis.  
- Improve adaptability by learning from past experiences.  
- Support consistency in behavior across various situations.  

...

### Generated Output

Revised output:  
- Enhance decision-making using historical data analysis.  
- Improve adaptability by learning from experiences.  
- Support consistency in behavior across situations.  

### Evaluation

Score: 9

APPROVED

## Approved after 2 iteration(s)!

## 5.6 Reasoning Topology Comparison

| Topology | Shape | Key Operation | Best For | LLM Calls |
|----------|-------|--------------|---------|----------|
| **CoT** | Linear chain | Step-by-step reasoning | Simple multi-step problems | 1 |
| **ToT** | Tree (beam search) | Branch → Score → Prune | Competitive path selection | 3-5+ |
| **GoT** | DAG (graph) | Branch → Contrast → Merge | Multi-perspective synthesis | 4-6+ |
| **AoT** | Planned DAG | Plan → Validate → Execute | Auditable production pipelines | 3+ |
| **ReAct** | Loop | Think → Act → Observe | Dynamic exploration with tools | Variable |
| **Self-Reflect** | Loop | Generate → Evaluate → Improve | Quality-critical tasks | 2-6 |

### When to Use Each

- **CoT**: Default for most tasks — simple and effective
- **ToT**: When you need to explore alternatives and pick the best
- **GoT**: When multiple perspectives need to be synthesized
- **AoT**: When execution must be deterministic and auditable
- **ReAct**: When the agent needs to interact with tools dynamically
- **Self-Reflect**: When output quality is critical and iteration is affordable

## 💡 Exercise 5: Build a Planner-Executor Agent

**Task**: Build a planner-executor system that:
1. Takes a research question
2. Creates a structured plan (Pydantic model)
3. Executes each step
4. Self-evaluates the final output

Test with: *"What are the key differences between LangGraph and CrewAI for building multi-agent systems?"*

In [ ]:
# Exercise 5: YOUR CODE HERE


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **CoT** | "Think step by step" dramatically improves LLM reasoning |
| **Task Decomposition** | Break complex tasks into structured sub-tasks with dependencies |
| **Planner-Executor** | Separate planning (LLM) from execution (code) for safety |
| **Tree of Thought** | Explore multiple paths, score and select the best |
| **Self-Reflection** | Generate → Evaluate → Improve loop for quality-critical output |

### What's Next

In **Notebook 06: Memory in AI Agents**, we give our agents the ability to remember — short-term conversation memory, long-term persistence, and context engineering strategies.